In [ ]:
import os
import re
import subprocess
from pathlib import Path

import anndata as ad
import scanpy as sc
import squidpy as sq

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, SHELL, activate_env_cmd, remove_tick_and_label

In [ ]:
# Settings
seed = 42

In [ ]:
mosta_dir = DATA_DIR / "MOSTA"
mosta_results = RESULT_DIR / "MOSTA"

script_dir = Path("batch_integration")

processed_path = mosta_results / "E16.5_E2.h5ad"

In [ ]:
mosta_results.mkdir(exist_ok=True, parents=True)

# Prepare dataset

In [ ]:
def prepare_adata(file: Path):
    adata = ad.read_h5ad(file)

    adata.var = adata.var[[]]
    adata.obs = adata.obs[["annotation"]]
    adata.X = adata.layers["count"]

    del adata.layers
    del adata.varm
    del adata.uns
    return adata

In [ ]:
# sorting files first guarantees that the AnnData will be ordered
files = {
    int(re.search(r"_E\d+S(\d+)\.", file.stem).groups()[0]): file
    for file in mosta_dir.glob("E16.5_E2S*.MOSTA.h5ad")
}
files = dict(sorted(files.items(), key=lambda item: item[0]))

In [ ]:
adata = ad.concat(
    {i: prepare_adata(file) for i, file in files.items()},
    index_unique="_",
    label="section",
)

adata.uns["batch_keys"] = ["section"]

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_cells(adata, min_counts=250)

preprocess_genes(adata, min_cells=200, batch_key="section")
preprocess_scaling(adata, n_pcs=50, seed=seed)

adata

In [ ]:
adata.write_h5ad(processed_path)

# Batch integration

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

In [ ]:
methods = [
    "no_batch_integration",
    "harmony",
    "bbknn",
    "scvi",
]

In [ ]:
for method in methods:
    conda_cmd = activate_env_cmd(method)

    script_path = script_dir / f"{method}.py"
    out_file = mosta_results / f"{method}.npz"

    cmd = f"{script_path} {processed_path} {out_file}"

    # because bbknn uses n neighbors per batch we derease the number of neighbors
    # so its roughly equal to the rest
    if method == "bbknn":
        n = 3  # too high?
        cmd += f" --n_neighs {n}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Clustering

In [ ]:
import time

from scipy.sparse import load_npz
from spatialleiden import spatialleiden

In [ ]:
adata = ad.read_h5ad(processed_path)


sq.gr.spatial_neighbors(
    adata, library_key=adata.uns["batch_keys"][0], coord_type="grid", n_neighs=4
)

In [ ]:
for file in mosta_results.glob("*.npz"):
    method = file.stem

    print(f"Processing {method}")

    adata.obsp["connectivities"] = load_npz(file)

    start = time.time()
    spatialleiden(
        adata,
        resolution=(1, 1),
        directed=(False, False),
        layer_ratio=1.5,  # TODO: change for bbknn?
        n_iterations=3,
    )
    end = time.time()
    print(f"Time: {(end - start) / 60:.2f} min")

    columns = ["spatialleiden"] + list(adata.uns["batch_keys"])

    labels = adata.obs[columns]
    labels.loc[:, ["x", "y"]] = adata.obsm["spatial"]
    labels.to_parquet(mosta_results / f"{method}.parquet")

# Plotting

In [ ]:
import pandas as pd
import seaborn as sns

scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

In [ ]:
import matplotlib.pyplot as plt

# Don't show figure
_ = plt.ioff()

In [ ]:
def plot_sections(label_file, batch_key: str = "section"):
    labels = pd.read_parquet(label_file).astype({"spatialleiden": "category"})

    g = sns.FacetGrid(
        labels, col=batch_key, col_wrap=4, legend_out=False, sharex=False, sharey=False
    )
    _ = g.map_dataframe(
        sns.scatterplot,
        x="x",
        y="y",
        hue="spatialleiden",
        legend=False,
        **scatter_kwargs,
    )

    for ax in g.axes.flat:
        ax.set(aspect=1)
        remove_tick_and_label(ax)

    _ = g.tight_layout()

    return g

## No integration

In [ ]:
fig = plot_sections(mosta_results / "no_batch_integration.parquet")
fig.savefig(mosta_results / "no_batch_integration.png")

## Harmony

In [ ]:
fig = plot_sections(mosta_results / "harmony.parquet")
fig.savefig(mosta_results / "harmony.png")

## scVI

In [ ]:
fig = plot_sections(mosta_results / "scvi.parquet")
fig.savefig(mosta_results / "scvi.png")

## bbknn

In [ ]:
fig = plot_sections(mosta_results / "bbknn.parquet")
fig.savefig(mosta_results / "bbknn.png")